# CUAD Fine-tune — extractive-QA on 41 clause types (Colab GPU)

Runs on Colab with a GPU runtime. Fine-tunes a DeBERTa-v3-base extractive-QA model on
the CUAD dataset, tracks to a local MLflow file-store, saves a self-contained bundle,
and registers it as `cuad-extractor` in the MLflow model registry.

**Runtime:** `Runtime > Change runtime type > GPU (T4 or better)`.

## 1. Install

Install the package from the merged `master` branch (or swap in your feature branch name).

In [ ]:
# Use a GPU runtime: Runtime > Change runtime type > GPU.
# --no-cache-dir avoids pip serving a stale cached build (the version stays 0.1.0).
# Do NOT use --force-reinstall here: it drags every dependency (numpy/torch/...) and
# clashes with Colab's preinstalled stack, leaving mismatched, broken installs.
!pip install -q --no-cache-dir "docintel[train,kie] @ git+https://github.com/KhoiDang1209/AI-Document-Understanding.git@master#subdirectory=docintel"
# Colab's torchaudio is built for a different CUDA than its torch; QA training doesn't use it, so drop it.
!pip uninstall -y -q torchaudio

## 2. Setup

In [ ]:
import subprocess
from pathlib import Path

import mlflow

from docintel.contracts.qa_config import QaTrainingConfig
from docintel.contracts.train_qa import run_qa_training

# 1 epoch + batch 32 + bf16 (config default) keeps the run within a Colab Pro session.
config = QaTrainingConfig(num_train_epochs=1.0, train_batch_size=32)
mlflow.set_tracking_uri("file:./mlruns")  # local file-store on the Colab VM
mlflow.set_experiment("cuad-qa")

In [ ]:
# Checkpoint fallback: save to Google Drive so a timed-out session can resume.
# Mount Drive, reuse the ocr-checkpoints folder if it exists, otherwise create it.
# run_qa_training() auto-detects the latest checkpoint here and resumes from it.
from google.colab import drive

drive.mount("/content/drive")
CHECKPOINT_DIR = "/content/drive/MyDrive/ocr-checkpoints"
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
print("Checkpoints ->", CHECKPOINT_DIR)

## 3. Load dataset

In [ ]:
from datasets import load_dataset

DATASET_REVISION = "main"  # pin for reproducibility
# trust_remote_code=True: CUAD ships a loader script (datasets 3.x requires opting in).
raw = load_dataset("theatticusproject/cuad-qa", revision=DATASET_REVISION, trust_remote_code=True)
print(raw)

## 4. Tokenize (SQuAD-style sliding window)

CUAD contracts are long documents. We use `doc_stride` and `max_seq_length` from
`QaTrainingConfig` (defaults: `stride=128`, `max_seq_length=512`) with
`return_overflowing_tokens=True` so each contract is split into overlapping windows.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(config.model_name)


def _tokenize_split(examples):
    """Tokenize (question, context) pairs with sliding-window overflow."""
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=config.max_seq_length,
        stride=config.doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    # Carry over answer positions from the parent example via sample_mapping
    sample_map = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions, end_positions = [], []
    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answers = examples["answers"][sample_idx]
        if len(answers["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue
        answer_start = answers["answer_start"][0]
        answer_end = answer_start + len(answers["text"][0])
        # Find token indices that contain the answer
        token_start = token_end = 0
        for idx, (start, end) in enumerate(offsets):
            if start <= answer_start < end:
                token_start = idx
            if start < answer_end <= end:
                token_end = idx
        start_positions.append(token_start)
        end_positions.append(token_end)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized


train_ds = raw["train"].map(_tokenize_split, batched=True, remove_columns=raw["train"].column_names)
eval_ds = raw["test"].map(_tokenize_split, batched=True, remove_columns=raw["test"].column_names)
print(f"train windows: {len(train_ds)}, eval windows: {len(eval_ds)}")

## 5. Train & track

In [ ]:
git_sha = (
    subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True
    ).stdout.strip()
    or "unknown"
)
bundle = run_qa_training(
    config=config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    bundle_dir=Path("cuad-extractor-bundle"),
    dataset_revision=DATASET_REVISION,
    git_sha=git_sha,
    output_dir=CHECKPOINT_DIR,  # checkpoints to Drive; resumes here after a timeout
)
print("Bundle saved to:", bundle)

## 6. Register in MLflow model registry

In [ ]:
# The last run logged the bundle as an artifact; register it.
runs = mlflow.search_runs(experiment_names=["cuad-qa"], order_by=["start_time DESC"])
run_id = runs.iloc[0]["run_id"]
artifact_uri = f"runs:/{run_id}/bundle"
mv = mlflow.register_model(artifact_uri, "cuad-extractor")
print(f"Registered cuad-extractor version {mv.version}")

## 7. Next step

Run `notebooks/cuad_onnx_export.ipynb` to download this bundle, export to ONNX fp32,
quantize to INT8, evaluate, and register `cuad-extractor-onnx-int8`.

To serve locally on the laptop set:
```
DOCINTEL_CONTRACT_ONNX_LOCAL_PATH=/path/to/cuad-int8-bundle
```
then start the API and `POST /contracts/extract` with a contract PDF.